# 손 키포인트 모델 → RKNN 변환 (ROCK 4D / RK3576) — 공식 경로

**x86 PC 없이 Google Colab(무료 x86 리눅스)에서 실행하세요.** (런타임: CPU면 충분)

> ⚠️ Ultralytics 의 `format=rknn` 네이티브 export 는 **detection 전용 (pose 미지원)** 입니다.
> 그래서 Rockchip 공식 방식인 **airockchip 포크 export + rknn_model_zoo 후처리** 를 따릅니다.
> 이 포크는 헤드를 분리하고 DFL/디코드를 제거해 NPU 양자화·속도를 개선합니다 (디코드는 보드 CPU에서).

흐름:  사전학습 `best.pt`(손 21키포인트) → **airockchip** 포크로 분리브랜치 ONNX → `hand_pose.rknn` (rk3576)

## 1) 우리가 학습한 손 키포인트 best.pt 업로드 (100ep, mAP50(P)≈0.87)

In [ ]:
# 우리가 직접 학습한 best.pt 업로드
#   PC 경로: Desktop\\hand_train\\runs\\pose\\hand_pose\\weights\\best.pt
from google.colab import files
import os
up = files.upload()                       # 파일 선택창에서 best.pt 선택
fn = list(up.keys())[0]
os.replace(fn, 'best.pt')
print('best.pt', os.path.getsize('best.pt'), 'bytes (우리 학습 모델)')

## 2) airockchip YOLO11 포크로 RKNN 최적화 ONNX 내보내기
이 포크(`ultralytics_yolo11`)의 `exporter.py` 가 헤드를 분리한 ONNX(detection 브랜치 + keypoint 브랜치)를
만듭니다. YOLO11 모델 전용 포크라 우리가 학습한 yolo11n-pose best.pt 를 그대로 처리합니다.

In [ ]:
%cd /content
!rm -rf ultralytics_yolo11 best.onnx          # 업로드한 best.pt 는 보존(삭제 안 함)
!git clone -q https://github.com/airockchip/ultralytics_yolo11.git
%cd /content/ultralytics_yolo11
!pip install -q onnx onnxslim onnxscript   # torch 2.11 onnx export 가 onnxscript 요구
!cp /content/best.pt ./best.pt

# default.yaml: model/task/batch/opset/imgsz(=보드 프로필 640)
import re
p='ultralytics/cfg/default.yaml'; s=open(p).read()
def setkv(s,k,v):
    if re.search(rf'^{k}:.*$', s, flags=re.M):
        return re.sub(rf'^{k}:.*$', f'{k}: {v}', s, count=1, flags=re.M)
    return s + f'\n{k}: {v}\n'
for k,v in [('model','best.pt'),('task','pose'),('batch','1'),('opset','12'),('imgsz','640')]:
    s=setkv(s,k,v)
open(p,'w').write(s)
print('default.yaml 설정 완료 (opset=12, imgsz=640)')

!export PYTHONPATH=./ && python ./ultralytics/engine/exporter.py
import glob; print('생성된 ONNX:', glob.glob('/content/ultralytics_yolo11/*.onnx'))

In [ ]:
# ONNX 출력 텐서 형태 확인 — 경로 자동 탐색
import onnx, glob
cands = sorted(glob.glob('/content/ultralytics_yolo11/*.onnx')
               + glob.glob('/content/**/*.onnx', recursive=True))
print('찾은 ONNX:', cands)
assert cands, 'ONNX 없음 → 4번 export 셀 실패. 4번 셀을 다시 실행해 마지막 에러를 확인하세요.'
ONNX_PATH = cands[0]
for o in onnx.load(ONNX_PATH).graph.output:
    print(o.name, [d.dim_value for d in o.type.tensor_type.shape.dim])

## 3) ONNX → RKNN 변환 (rk3576)

> ⚠️ `rknn-toolkit2` 는 **numpy<2** 를 요구해 설치 시 numpy 를 다운그레이드합니다.
> Colab 커널엔 이미 numpy 2.x 가 로드돼 있어, 설치 직후 변환하면 `No module named 'numpy.char'` 에러가 납니다.
> **해결: 아래 설치 셀 실행 → 런타임 재시작(런타임 → 세션 다시 시작) → 변환 셀 실행.**
> best.onnx 는 디스크에 남아 있어 사라지지 않습니다.

In [ ]:
# [설치] rknn-toolkit2 (보드의 rknnlite 2.3.0 과 정합) — 실행 후 반드시 런타임 재시작!
!pip install -q rknn-toolkit2==2.3.0
print("설치 완료 → 상단 메뉴 '런타임 → 세션 다시 시작' 후, 아래 변환 셀을 실행하세요")

In [ ]:
# [변환] ← 런타임 재시작 후 이 셀부터 실행 (best.onnx 는 디스크에 남아 있음)
import os, glob
from rknn.api import RKNN
cands = sorted(glob.glob('/content/ultralytics_yolo11/*.onnx')
               + glob.glob('/content/**/*.onnx', recursive=True))
assert cands, 'best.onnx 없음 — 1~2 단계(2,4번 셀)를 먼저 실행하세요'
ONNX = cands[0]; print('사용할 ONNX:', ONNX)

def convert(onnx_path, out, target='rk3576', dataset=None):
    rknn = RKNN(verbose=False)
    # YOLO 입력: RGB uint8, /255 정규화 -> mean=0, std=255 (rknn_model_zoo 와 동일)
    rknn.config(mean_values=[[0,0,0]], std_values=[[255,255,255]], target_platform=target)
    assert rknn.load_onnx(model=onnx_path)==0, 'load_onnx 실패'
    do_quant = dataset is not None and os.path.exists(dataset)
    assert rknn.build(do_quantization=do_quant, dataset=dataset if do_quant else None)==0, 'build 실패'
    assert rknn.export_rknn(out)==0, 'export 실패'
    print('완료:', out, os.path.getsize(out), 'bytes', '(INT8)' if do_quant else '(FP16)')
    rknn.release()

# FP16: 보정 이미지 불필요, 정확도 안정적 (rk3576 NPU 가속). 우선 권장.
convert(ONNX, '/content/hand_pose_640.rknn', 'rk3576', dataset=None)

### (선택) INT8 양자화 — 더 빠름, 보정 이미지 필요
손이 나온 대표 이미지 20~200장을 `calib/` 에 넣고 `calib_list.txt` 를 만든 뒤 아래를 실행.
정확도가 떨어지면 위 FP16 모델을 쓰세요.

In [ ]:
# import glob
# open('calib_list.txt','w').write('\n'.join(sorted(glob.glob('calib/*.jpg'))))
# convert(ONNX, 'hand_pose_int8.rknn', 'rk3576', dataset='calib_list.txt')

## 4) 다운로드 → 보드의 `models/` 로 복사

In [ ]:
from google.colab import files
files.download('/content/hand_pose_640.rknn')

---
보드에 배포:
```bash
# 받은 hand_pose_640.rknn 을 보드로 복사 후
mv ~/Downloads/hand_pose_640.rknn ~/Documents/ai_curtain_control/models/
sudo systemctl restart ai-curtain      # 또는 python3 serve.py
```
> 활성 손 프로필(hand_near)이 models/hand_pose_640.rknn (imgsz 640)을 사용합니다.
> 후처리(src/postprocess.py)는 분리브랜치 출력(3 det + 1 kpt, CPU DFL 디코드)에 맞춰져 있습니다.